# Reward Modeling & RLHF Notebook

> Hands-on Build It and Exercises.

## Build It

This lesson uses tiny synthetic "prompts" and "responses" represented as strings. The RM is a linear scorer over a bag-of-tokens representation. No real LLM — the *shape* of the pipeline matters, not the scale. See `code/main.py`.

### Step 1: synthetic preference data

In [ ]:
```python

PROMPTS = ["help me", "answer me", "explain this"]

GOOD_WORDS = {"clear", "specific", "kind", "thorough"}

BAD_WORDS = {"vague", "rude", "wrong", "short"}

def make_pair(rng):

    x = rng.choice(PROMPTS)

    y_good = rng.choice(list(GOOD_WORDS)) + " " + rng.choice(list(GOOD_WORDS))

    y_bad = rng.choice(list(BAD_WORDS)) + " " + rng.choice(list(BAD_WORDS))

    return (x, y_good, y_bad)

In [ ]:
```

In real RLHF this is replaced by human labelers. The shape — `(prompt, preferred_response, rejected_response)` — is identical.

### Step 2: Bradley-Terry reward model

Linear score: `R(x, y) = w · bag(y)`. Train to minimize the BT pairwise log-loss:

In [ ]:
```python

def rm_train_step(w, x, y_pos, y_neg, lr):

    r_pos = dot(w, bag(y_pos))

    r_neg = dot(w, bag(y_neg))

    p = sigmoid(r_pos - r_neg)

    for tok, cnt in bag(y_pos).items():

        w[tok] += lr * (1 - p) * cnt

    for tok, cnt in bag(y_neg).items():

        w[tok] -= lr * (1 - p) * cnt

In [ ]:
```

After a few hundred updates, `w` assigns positive weights to good-word tokens and negative to bad.

### Step 3: PPO-like policy on top of RM

Our toy policy produces a single token from a vocabulary. We score the token under the RM, compute `log π_θ(token | prompt)`, add a KL-to-reference penalty, and apply the clipped PPO surrogate.

In [ ]:
```python

def rlhf_step(theta, ref, w, prompt, rng, eps=0.2, beta=0.1, lr=0.05):

    logits_theta = policy_logits(theta, prompt)

    probs = softmax(logits_theta)

    token = sample(probs, rng)

    logits_ref = policy_logits(ref, prompt)

    probs_ref = softmax(logits_ref)

    reward = dot(w, bag([token])) - beta * kl(probs, probs_ref)

    # ppo-style update on theta, treating reward as the return

    ...

In [ ]:
```

### Step 4: monitor the KL

Track mean `KL(π_θ || π_ref)` every update. If it creeps past `~5-10` the policy has drifted far from `π_SFT` — lower `β` is rising or reward hacking is starting. This is the top diagnostic in real RLHF.

### Step 5: the production recipe with TRL

Once you understand the toy pipeline, here is the same loop as a real library user writes it. Hugging Face's [TRL](https://huggingface.co/docs/trl) is the reference implementation — `RewardTrainer` for Stage 2 and `PPOTrainer` (with a KL-to-reference built in) for Stage 3.

In [ ]:
```python

# Stage 2: reward model from pairwise preferences

from trl import RewardTrainer, RewardConfig

from transformers import AutoModelForSequenceClassification, AutoTokenizer

tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

rm = AutoModelForSequenceClassification.from_pretrained(

    "meta-llama/Llama-3.1-8B-Instruct", num_labels=1

)

# dataset rows: {"prompt", "chosen", "rejected"} — Bradley-Terry format

trainer = RewardTrainer(

    model=rm,

    tokenizer=tok,

    train_dataset=preference_data,

    args=RewardConfig(output_dir="./rm", num_train_epochs=1, learning_rate=1e-5),

)

trainer.train()

In [ ]:
```

In [ ]:
```python

# Stage 3: PPO against the RM with KL penalty to the SFT reference

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

policy = AutoModelForCausalLMWithValueHead.from_pretrained("./sft-checkpoint")

ref    = AutoModelForCausalLMWithValueHead.from_pretrained("./sft-checkpoint")  # frozen

ppo = PPOTrainer(

    config=PPOConfig(learning_rate=1.41e-5, batch_size=64, init_kl_coef=0.05,

                     target_kl=6.0, adap_kl_ctrl=True),

    model=policy, ref_model=ref, tokenizer=tok,

)

for batch in dataloader:

    responses = ppo.generate(batch["query_ids"], max_new_tokens=128)

    rewards   = rm(torch.cat([batch["query_ids"], responses], dim=-1)).logits[:, 0]

    stats     = ppo.step(batch["query_ids"], responses, rewards)

    # stats includes: mean_kl, clip_frac, value_loss — the three PPO diagnostics

In [ ]:
```

Three things the library does for you. `adap_kl_ctrl=True` implements the adaptive-β schedule: if observed KL exceeds `target_kl`, β doubles; if below half, β halves. The reference model is frozen by convention — you must not accidentally share parameters with `policy`. And the value head lives on the same backbone as the policy (`AutoModelForCausalLMWithValueHead` attaches a scalar MLP head), which is why TRL reports `policy/kl` and `value/loss` separately.

## Exercises

In [ ]:
1. **Easy.** Train the Bradley-Terry reward model in `code/main.py` on 500 synthetic preference pairs. Measure pairwise accuracy on a held-out 100 pairs. Should exceed 90%.
2. **Medium.** Run the toy PPO-RLHF loop with `β ∈ {0.0, 0.1, 1.0}`. For each, plot RM score vs KL-to-reference over updates. Which runs reward-hack?
3. **Hard.** Implement DPO (closed-form preference-likelihood loss) on the same preference data and compare to the RLHF-PPO pipeline in compute used and final RM score achieved.